# RG3 저장 모델 예측 (재학습 없음)


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import nbformat
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rg3/01_preprocess_train.ipynb').exists())
n=nbformat.read(ROOT/'rg3/01_preprocess_train.ipynb',4)
exec(n.cells[1].source) # 정의만 실행. 학습 cell은 실행하지 않음.
INPUT_CSV=ROOT/'dataset/processed/rg3/rg3_unlabeled_input_features.csv' # Default: same6590 unknown shots
f=pd.read_csv(INPUT_CSV,parse_dates=['TimeStamp']).drop(columns=['PassOrFail','Reason'],errors='ignore')
gas_bundle=load_bundle(MODEL_DIR/'gas12.joblib')
import hashlib
model_path=MODEL_DIR/'shortshot_routed_D.joblib'
metadata=json.loads((MODEL_DIR/'shortshot_routed_D.json').read_text(encoding='utf-8'))
assert hashlib.sha256(model_path.read_bytes()).hexdigest()==metadata['model_sha256'],'Shortshot model hash mismatch'
assert metadata['sklearn_version']==sklearn.__version__,'Use matching sklearn '+metadata['sklearn_version']
short_bundle=joblib.load(model_path) # Trusted locally generated artifact only
out=predict_gas_features(gas_bundle,f)
short,details=predict_routed_short(short_bundle,f)
for name,value in details.items():out[name]=value
out['shortshot_candidate']=short.astype(int)
out['any_defect_candidate']=(out.gas_prediction.eq(1)|short).astype(int)
out['predicted_type']=np.select([out.status.ne('ok'),out.gas_prediction.eq(1)&short,out.gas_prediction.eq(1),short],['판정보류','가스+미성형 후보','가스 후보','미성형 후보'],default='경보 없음(정상 확정 아님)')
# 두 모델은 모두 실행하므로 하나가 양성이라고 다른 모델을 건너뛰지 않는다.
outdir=ROOT/'rg3/predictions';outdir.mkdir(exist_ok=True)
out.to_csv(outdir/'rg3_predictions.csv',index=False,encoding='utf-8-sig')
print('Saved-model inference; unknown truth / training samples not validation:',len(out))
print(out.predicted_type.value_counts())
display(out.head())


